# 🚗 Vehicle Detection Model Training
## Auto Hotel Luxor - Mexican Market

**Dataset:** CompCar filtered to 32 Mexican market brands (25K+ images)

**Steps:**
1. Setup GPU
2. Load dataset
3. Train MobileNetV3
4. Export to TFLite
5. Download model

---

### ⚙️ Setup

1. Ve a **Settings** (derecha) > **Accelerator** > **GPU T4**
2. En **Data** (izquierda) > **Add Data** > Sube `mexican_market.zip`
3. Ejecuta todas las celdas

In [ ]:
# 1. Verificar GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memoria: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    print('⚠️ Sin GPU! Ve a Settings > Accelerator > GPU T4')

In [ ]:
# 2. Instalar dependencias
!pip install -q tqdm

In [ ]:
# 3. Cargar dataset
# Si subiste el dataset como Kaggle Dataset, estará en /kaggle/input/
# Si lo subiste directamente, estará en /kaggle/working/

import zipfile
import os

# Buscar el zip
zip_paths = [
    '/kaggle/input/mexican-market/mexican_market.zip',
    '/kaggle/input/mexican_market.zip',
    '/kaggle/working/mexican_market.zip',
    'mexican_market.zip'
]

zip_path = None
for p in zip_paths:
    if os.path.exists(p):
        zip_path = p
        break

if zip_path:
    print(f'Found: {zip_path}')
    print('Extracting...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/kaggle/working/')
    print('Done!')
else:
    print('⚠️ Dataset not found!')
    print('Upload mexican_market.zip to Kaggle:')
    print('  1. Go to Data (left panel) > Add Data > Upload')
    print('  2. Select mexican_market.zip')
    print('  3. Run this cell again')

# Verificar
dataset_path = '/kaggle/working/mexican_market'
if os.path.exists(dataset_path):
    brands = os.listdir(f'{dataset_path}/train')
    total = sum(len(os.listdir(f'{dataset_path}/train/{b}')) for b in brands)
    print(f'\n✓ Dataset: {len(brands)} brands, {total} training images')

## 4. Train Model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import json
import time

class VehicleDataset(Dataset):
    def __init__(self, root, transform=None):
        self.images = []
        self.labels = []
        self.class_to_idx = {}
        
        root = Path(root)
        for idx, class_dir in enumerate(sorted([d for d in root.iterdir() if d.is_dir()])):
            self.class_to_idx[class_dir.name] = idx
            for img in class_dir.glob('*.jpg'):
                self.images.append(img)
                self.labels.append(idx)
        
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}
        self.num_classes = len(self.class_to_idx)
        self.transform = transform
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load
print('Loading datasets...')
train_ds = VehicleDataset(f'{dataset_path}/train', train_transform)
val_ds = VehicleDataset(f'{dataset_path}/val', val_transform)
print(f'Train: {len(train_ds)} images, {train_ds.num_classes} classes')
print(f'Val: {len(val_ds)} images')
print(f'\nBrands ({train_ds.num_classes}):')
for brand in sorted(train_ds.class_to_idx.keys()):
    count = len([l for l in train_ds.labels if l == train_ds.class_to_idx[brand]])
    print(f'  {brand}: {count}')

In [ ]:
# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = models.mobilenet_v3_small(pretrained=True)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, train_ds.num_classes)
model = model.to(device)

# DataLoaders (pin_memory for GPU)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

print(f'\nModel: MobileNetV3-Small')
print(f'Classes: {train_ds.num_classes}')
print(f'Batch: 64 | LR: 0.001')

In [ ]:
# Train
EPOCHS = 15
best_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print(f'Training for {EPOCHS} epochs...\n')
start_time = time.time()

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.1f}%'})
    
    train_acc = 100. * correct / total
    
    # Validate
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_acc = 100. * val_correct / val_total
    scheduler.step(val_acc)
    
    history['train_loss'].append(train_loss / len(train_loader))
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss / len(val_loader))
    history['val_acc'].append(val_acc)
    
    print(f'Epoch {epoch+1}: Train={train_acc:.1f}% Val={val_acc:.1f}%')
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc': val_acc,
            'num_classes': train_ds.num_classes,
            'class_to_idx': train_ds.class_to_idx,
            'idx_to_class': train_ds.idx_to_class,
        }, 'best_model.pth')
        print(f'  ✓ Saved (acc: {val_acc:.1f}%)')

elapsed = time.time() - start_time
print(f'\n{"="*50}')
print(f'✓ Training complete!')
print(f'Best accuracy: {best_acc:.1f}%')
print(f'Time: {elapsed/60:.1f} minutes')
print(f'{"="*50}')

## 5. Export to TFLite

In [ ]:
# Export to ONNX
!pip install -q onnx

import torch.onnx
import json

# Load best model
checkpoint = torch.load('best_model.pth', map_location='cpu')
model = models.mobilenet_v3_small(pretrained=False)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, checkpoint['num_classes'])
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Export ONNX
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(model, dummy, 'vehicle_classifier.onnx',
    export_params=True, opset_version=11,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}})
print('✓ ONNX exported')

# Save labels
labels = {str(k): v for k, v in checkpoint['idx_to_class'].items()}
with open('vehicle_labels.json', 'w') as f:
    json.dump(labels, f, indent=2)
print('✓ Labels saved')

# TFLite conversion
try:
    import tensorflow as tf
    import onnx
    from onnx_tf.backend import prepare
    
    onnx_model = onnx.load('vehicle_classifier.onnx')
    tf_rep = prepare(onnx_model)
    tf_rep.export_graph('saved_model')
    
    converter = tf.lite.TFLiteConverter.from_saved_model('saved_model')
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    
    with open('vehicle_classifier.tflite', 'wb') as f:
        f.write(tflite_model)
    print(f'✓ TFLite exported ({len(tflite_model)/1024/1024:.1f} MB)')
except Exception as e:
    print(f'TFLite: {e}')
    print('Download ONNX and convert locally')

## 6. Download Model

In [ ]:
import os

print('Model files ready for download:')
print('='*50)

files_to_download = [
    ('best_model.pth', 'PyTorch model'),
    ('vehicle_labels.json', 'Class labels'),
    ('vehicle_classifier.onnx', 'ONNX model'),
]

for fname, desc in files_to_download:
    if os.path.exists(fname):
        size = os.path.getsize(fname) / 1024 / 1024
        print(f'✓ {fname} ({size:.1f} MB) - {desc}')

if os.path.exists('vehicle_classifier.tflite'):
    size = os.path.getsize('vehicle_classifier.tflite') / 1024 / 1024
    print(f'✓ vehicle_classifier.tflite ({size:.1f} MB) - TFLite')

print('\n' + '='*50)
print('\nTo download:')
print('1. Go to Output (right panel)')
print('2. Click on each file to download')
print('3. Copy to: ml-models/exported/')
print('\nOr use this code:')
print()
print('from IPython.display import FileLink')
print('FileLink("best_model.pth")')
print('FileLink("vehicle_labels.json")')
print('FileLink("vehicle_classifier.onnx")')

In [ ]:
# Quick download links
from IPython.display import FileLink, display

print('Click to download:\n')
for f in ['best_model.pth', 'vehicle_labels.json', 'vehicle_classifier.onnx']:
    if os.path.exists(f):
        display(FileLink(f))

## 7. Test Inference

In [ ]:
# Test
import random
from PIL import Image

checkpoint = torch.load('best_model.pth', map_location='cpu')
model = models.mobilenet_v3_small(pretrained=False)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, checkpoint['num_classes'])
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

idx_to_class = checkpoint['idx_to_class']

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print('Testing 15 random images...\n')

test_images = random.sample(val_ds.images, 15)
correct = 0

for img_path in test_images:
    img = Image.open(img_path).convert('RGB')
    inp = test_transform(img).unsqueeze(0)
    
    with torch.no_grad():
        out = model(inp)
        _, pred = out.max(1)
    
    predicted = idx_to_class[pred.item()]
    actual = img_path.parent.name
    match = '✓' if predicted == actual else '✗'
    if predicted == actual:
        correct += 1
    
    print(f'{match} {actual:15s} → {predicted}')

print(f'\nAccuracy: {correct}/15 ({100*correct/15:.0f}%)')